<div style="color:#3c4d5a; border-top: 7px solid #42A5F5; border-bottom: 7px solid #42A5F5; padding: 5px; text-align: center; text-transform: uppercase"><h1>Entrenamiento Modelos - Predicción</h1> </div>


<div id="RNN" style="color:#37475a; border-bottom: 4px solid #4E853C; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Importar Modulos</h2> </div>

In [1]:
import pandas as pd 
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import SimpleRNN, Dense 
from tensorflow.keras.callbacks import EarlyStopping




<div id="RNN" style="color:#37475a; border-bottom: 4px solid #4E853C; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Carga Dataset</h2> </div>

In [2]:
df = pd.read_csv('data/sales_history.csv')
df['fecha_venta'] = pd.to_datetime(df['fecha_venta'])



<div id="RNN" style="color:#37475a; border-bottom: 4px solid #4E853C; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Feature Engineering: Variables temporales</h2> </div>

In [ ]:
num_dias = 60

productos = [
    {"id": 101, "nombre": "Leche Entera 1L", "cat": "Lácteos", "un": "Caja", "st_ini": 500, "tasa_base": 30},
    {"id": 102, "nombre": "Arroz Blanco 1kg", "cat": "Granos", "un": "Bolsa", "st_ini": 1200, "tasa_base": 45},
    {"id": 103, "nombre": "Aceite Vegetal 900ml", "cat": "Abarrotes", "un": "Botella", "st_ini": 300, "tasa_base": 15},
    {"id": 104, "nombre": "Pasta 500g", "cat": "Abarrotes", "un": "Paquete", "st_ini": 450, "tasa_base": 20},
    {"id": 105, "nombre": "Café Molido 250g", "cat": "Bebidas", "un": "Sobre", "st_ini": 200, "tasa_base": 10},
    {"id": 106, "nombre": "Detergente Líquido 3L", "cat": "Limpieza", "un": "Bidón", "st_ini": 150, "tasa_base": 5},
    {"id": 107, "nombre": "Atún en Conserva 140g", "cat": "Enlatados", "un": "Lata", "st_ini": 800, "tasa_base": 35},
    {"id": 108, "nombre": "Huevos (Docena)", "cat": "Fresco", "un": "Cartón", "st_ini": 100, "tasa_base": 12},
    {"id": 109, "nombre": "Harina de Trigo 1kg", "cat": "Granos", "un": "Bolsa", "st_ini": 600, "tasa_base": 18},
    {"id": 110, "nombre": "Papel Higiénico 4u", "cat": "Higiene", "un": "Pack", "st_ini": 400, "tasa_base": 25},
]

sales_data = []
inventory_state = {p['id']: p['st_ini'] for p in productos}
fecha_inicio = datetime(2026, 3, 7)

for i in range(num_dias):
    fecha = fecha_inicio + timedelta(days=i)
    is_weekend = fecha.weekday() >= 5
    
    for p in productos:
        venta_proyectada = int(np.random.poisson(lam=p['tasa_base'] * (1.5 if is_weekend else 1.0)))
        
        stock_en_momento = inventory_state[p['id']]
        venta_real = min(venta_proyectada, stock_en_momento)
        
        sales_data.append([p['id'], p['nombre'], fecha.strftime('%Y-%m-%d'), venta_real, stock_en_momento])
        nuevo_stock = stock_en_momento - venta_real
        if nuevo_stock < (p['st_ini'] * 0.15): 
            nuevo_stock += p['st_ini'] // 2   
            
        inventory_state[p['id']] = nuevo_stock

df_sales = pd.DataFrame(sales_data, columns=['product_id', 'nombre', 'fecha_venta', 'cantidad_vendida', 'stock_en_momento'])
df_sales.to_csv('data/sales_history.csv', index=False)

df_inventory = pd.DataFrame([[p['id'], p['nombre'], p['cat'], p['un'], inventory_state[p['id']], '2025-12-31'] for p in productos], 
                             columns=['product_id', 'nombre', 'categoria', 'unidad', 'stock_actual', 'fecha_expiracion'])
df_inventory.to_csv('data/inventory.csv', index=False)

In [3]:
df['dia_semana'] =  df['fecha_venta'].dt.dayofweek
df['mes'] = df['fecha_venta'].dt.month
df['fin_de_semana'] = (df['dia_semana'] >=5).astype(int)
df.head()

,product_id,nombre,fecha_venta,cantidad_vendida,stock_en_momento,dia_semana,mes,fin_de_semana
0,101,Leche Entera 1L,2024-03-05,9,500,1,3,0
1,102,Arroz Blanco 1kg,2024-03-05,9,1200,1,3,0
2,103,Aceite Vegetal 900ml,2024-03-05,16,300,1,3,0
3,104,Pasta 500g,2024-03-05,5,450,1,3,0
4,105,Café Molido 250g,2024-03-05,12,200,1,3,0




<div id="RNN" style="color:#37475a; border-bottom: 4px solid #4E853C; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Normalización</h2> </div>

In [4]:
columnas  = ['cantidad_vendida', 'dia_semana', 'mes', 'fin_de_semana']
scalar_y = MinMaxScaler().fit(df[['cantidad_vendida']])
df[columnas] = MinMaxScaler().fit_transform(df[columnas])

In [5]:
pasos = 7
X, y = [], []

for pid in df['product_id'].unique():
    datos = df[df['product_id'] == pid][columnas].values
    ventas = df[df['product_id'] == pid]['cantidad_vendida'].values
    for i in range(len(datos)-pasos):
        X.append(datos[i : i + pasos])
        y.append(ventas[i + pasos])

X, y = np.array(X), np.array(y)

split = int(len(X) * 0.7)

X_train, X_test, y_train, y_test = X[:split], X[split:], y[:split], y[split:]

modelo = Sequential([
    SimpleRNN(32, activation='relu', input_shape=(pasos, len(columnas))),
    Dense(16, activation='relu'),
    Dense(1)
])

modelo.compile(optimizer='adam', loss='mse')

e_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

modelo.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test), callbacks=[e_stop], batch_size=16, verbose=0)

predic = scalar_y.inverse_transform(modelo.predict(X_test, verbose=0))
ytest_r= scalar_y.inverse_transform(y_test.reshape(-1,1))

rmse =  np.sqrt(mean_squared_error(ytest_r, predic))

print(f"Modelo entrenado. RMSE en test: {rmse:.2f}")

C:\Users\Usuario\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo entrenado. RMSE en test: 3.61



<div id="RNN" style="color:#37475a; border-bottom: 4px solid #4E853C; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Funciones Prediccion </h2> </div>

In [6]:
def predecir_demanda(product_id, fecha):
    fecha = pd.to_datetime(fecha)
    
    # 1. Filtrar el df hasta ANTES de la fecha objetivo
    hist_producto = df[
        (df['product_id'] == product_id) & 
        (df['fecha_venta'] < fecha)          
    ].sort_values('fecha_venta')
    
    # 2. Verificar que hay suficiente historial
    if len(hist_producto) < pasos:
        raise ValueError(f"No hay suficiente historial para {product_id} antes de {fecha}")
    
    # 3. Tomar los últimos `pasos` registros antes de esa fecha
    historial = hist_producto[columnas].values[-pasos:]
    
    # 4. Reconstruir features de fecha para el día a predecir
    dia_semana_norm  = fecha.dayofweek
    mes_norm         = fecha.month
    fin_semana_norm  = int(fecha.dayofweek >= 5)
    
    X_pred = historial.reshape(1, pasos, len(columnas))
    preds  = modelo.predict(X_pred, verbose=0)
    predr  = scalar_y.inverse_transform(preds)[0][0]
    return max(0, int(round(predr)))


def predecir_demanda_lista(lista, fecha):
    return {pid: predecir_demanda(pid, fecha) for pid in lista}

In [8]:
fecha_test = '2026-05-10'
print(f"\nPredicción individual (Prod 101): {predecir_demanda(101, fecha_test)} unidades")
print(f"Predicción en listado (Prod 101, 102, 103): {predecir_demanda_lista([101, 102, 103], fecha_test)}")


Predicción individual (Prod 101): 11 unidades
Predicción en listado (Prod 101, 102, 103): {101: 11, 102: 14, 103: 14}
